In [1]:
import os
import pandas as pd
import numpy as np
import requests
from PIL import Image
from io import BytesIO
import matplotlib.pyplot as plt

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms

from sklearn.impute import SimpleImputer
from sklearn.preprocessing import MinMaxScaler, OneHotEncoder
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_absolute_error, median_absolute_error, r2_score, mean_squared_error
from tqdm import tqdm

plt.style.use("seaborn-v0_8")
SEED = 42
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
device


device(type='cpu')

In [2]:
def sklearn_metrics_log(preds_log: np.ndarray, targets_log: np.ndarray):
    rmse = mean_squared_error(targets_log, preds_log, squared=False)
    mae = mean_absolute_error(targets_log, preds_log)
    medae = median_absolute_error(targets_log, preds_log)
    r2 = r2_score(targets_log, preds_log)
    return rmse, mae, medae, r2


In [19]:
from sklearn.feature_selection import VarianceThreshold


class AirbnbPreprocessorAndTrainer:
    def __init__(self, csv_path, nrows=1000, image_size=32, batch_size=64, lr=0.001, patience=5, seed=SEED):
        self.csv_path = csv_path
        self.nrows = nrows
        self.image_size = image_size
        self.batch_size = batch_size
        self.lr = lr
        self.patience = patience
        self.seed = seed

        self.device = device
        print("Verwende Gerät:", self.device)

        self.scaler = MinMaxScaler()
        self.imputer = SimpleImputer(strategy='median')
        self.encoder = None
        self.model = None

        self.image_column = "picture_url"
        self.target_column = "price"
        self.optional_id_col = "id"

        self.train_loader = None
        self.test_loader = None
        self.train_dataset = None
        self.test_dataset = None
        self.df = None
        self.images = None
        self.feature_names = None

    def preprocess(self):
    # 1) Laden und Grundprüfungen
        df = pd.read_csv(self.csv_path, nrows=self.nrows)
        if self.target_column not in df.columns:
            raise ValueError(f"Zielspalte '{self.target_column}' fehlt.")
        if self.image_column not in df.columns:
            raise ValueError(f"Bildspalte '{self.image_column}' fehlt.")

        # 2) Ziel und Bild vorhanden und bereinigt
        df = df.dropna(subset=[self.target_column, self.image_column]).copy()
        df[self.target_column] = (
            df[self.target_column]
            .astype(str)
            .str.replace(r"[\$\€\£,]", "", regex=True)
        )
        df[self.target_column] = pd.to_numeric(df[self.target_column], errors="coerce")
        df = df[df[self.target_column].notna()].copy()

        # Log-Transform
        df[self.target_column] = np.log(df[self.target_column])

        # Binäre Mappings, falls vorhanden
        if "host_is_superhost" in df.columns:
            df["host_is_superhost"] = df["host_is_superhost"].map(
                {'t': 1, 'f': 0, True: 1, False: 0}
            ).astype(float)
        if "host_identity_verified" in df.columns:
            df["host_identity_verified"] = df["host_identity_verified"].map(
                {'t': 1, 'f': 0, True: 1, False: 0}
            ).astype(float)

        # 3) Feature-Spalten bestimmen
        exclude = {self.image_column, self.target_column, "description"}
        if self.optional_id_col in df.columns:
            exclude.add(self.optional_id_col)
        feature_cols = [c for c in df.columns if c not in exclude]

        # numerisch vs. kategorial
        num_cols = [c for c in feature_cols if pd.api.types.is_numeric_dtype(df[c])]
        cat_cols = [c for c in feature_cols if c not in num_cols]

        parts = []

        # 4) Numerische imputen – exakt dieselben Zeilen/Index behalten
        # Numerische Spalten verarbeiten
        if len(self.num_cols) > 0:
            num_block = df[self.num_cols].copy()
            
            # Debugging: Anzahl Spalten vor Preprocessing
            print(f"Numerische Spalten vor Preprocessing: {len(num_block.columns)}")
            
            # Variance Threshold anwenden
            variance_selector = VarianceThreshold()
            num_var_filtered = variance_selector.fit_transform(num_block)
            selected_cols = num_block.columns[variance_selector.get_support()]
            
            print(f"Spalten nach VarianceThreshold: {len(selected_cols)}")
            print(f"Entfernte Spalten: {num_block.columns[~variance_selector.get_support()].tolist()}")
            
            # DataFrame mit gefilterten Spalten erstellen
            num_block_filtered = pd.DataFrame(num_var_filtered, index=num_block.index, columns=selected_cols)
            
            # Imputer anwenden
            num_imputed = self.imputer.fit_transform(num_block_filtered)
            
            # Sicherstellen dass Dimensionen übereinstimmen
            assert num_imputed.shape[1] == len(selected_cols), f"Dimension mismatch: {num_imputed.shape[1]} vs {len(selected_cols)}"
            
            num_df = pd.DataFrame(num_imputed, index=df.index, columns=selected_cols)
            parts.append(num_df)

        # 5) Kategoriale OHE – dichte Ausgabe, gleichbleibender Index
        if len(cat_cols) > 0:
            cat_block = df[cat_cols].astype("object").fillna("Unknown").copy()
            try:
                self.encoder = OneHotEncoder(sparse_output=False, handle_unknown="ignore")
            except TypeError:
                self.encoder = OneHotEncoder(sparse=False, handle_unknown="ignore")
            cat_mat = self.encoder.fit_transform(cat_block)
            cat_names = self.encoder.get_feature_names_out(cat_block.columns)
            cat_df = pd.DataFrame(cat_mat, index=df.index, columns=cat_names)
            parts.append(cat_df)

        if len(parts) == 0:
            raise ValueError("Keine nutzbaren Feature-Spalten gefunden.")

        # 6) Feature-Matrix bauen – gleicher Index garantiert
        feats_df = pd.concat(parts, axis=1)
        # Skalieren
        feats_scaled = self.scaler.fit_transform(feats_df.values)
        feats_df = pd.DataFrame(feats_scaled, index=feats_df.index, columns=feats_df.columns)
        self.feature_names = list(feats_df.columns)

        # 7) Keep-Columns und sauberes Zusammenführen
        keep_cols = [self.image_column, self.target_column]
        if self.optional_id_col in df.columns:
            keep_cols.append(self.optional_id_col)
        # concat auf gleichem Index, dann sauber reset_index
        merged = pd.concat([feats_df, df[keep_cols]], axis=1)
        merged = merged.loc[~merged[self.image_column].isna()].copy()

        # finaler, kompakter Index 0..N-1 für konsistente iloc
        self.df = merged.reset_index(drop=True)




    def process_images(self):
        images = []
        valid_indices = []
        for i, row in tqdm(self.df.iterrows(), total=len(self.df), desc="Bilder verarbeiten"):
            try:
                url = row[self.image_column]
                resp = requests.get(url, timeout=5)
                resp.raise_for_status()
                if 'image' not in resp.headers.get('Content-Type', ''):
                    raise ValueError("Kein Bild-Content")
                img = Image.open(BytesIO(resp.content)).convert("RGB")
                img = img.resize((self.image_size, self.image_size))
                images.append(np.array(img))
                valid_indices.append(i)
            except Exception as e:
                print(f"Fehler bei Index {i}: {e}")
        self.images = np.array(images)
        self.df = self.df.iloc[valid_indices].reset_index(drop=True)

    def prepare_tensors(self):
        exclude = {self.image_column, self.target_column}
        if self.optional_id_col in self.df.columns:
            exclude.add(self.optional_id_col)
        feature_cols = [c for c in self.df.columns if c not in exclude]
        self.feature_names = feature_cols

        X_tab = self.df[feature_cols].values.astype(np.float32)
        y = self.df[self.target_column].values.astype(np.float32)

        (X_train_tab, X_test_tab,
         X_train_img, X_test_img,
         y_train, y_test) = train_test_split(
            X_tab, self.images, y, test_size=0.2, random_state=self.seed
        )

        self.train_dataset = self.AirbnbDataset(X_train_img, X_train_tab, y_train)
        self.test_dataset  = self.AirbnbDataset(X_test_img, X_test_tab, y_test)

        self.train_loader = DataLoader(self.train_dataset, batch_size=self.batch_size, shuffle=True)
        self.test_loader  = DataLoader(self.test_dataset,  batch_size=self.batch_size, shuffle=False)

    class AirbnbDataset(Dataset):
        def __init__(self, images, tab_features, prices, transform=None):
            self.images = images
            self.tab_features = tab_features
            self.prices = prices
            self.transform = transform or transforms.Compose([
                transforms.ToTensor(),
                transforms.Normalize(mean=[0.485, 0.456, 0.406],
                                     std=[0.229, 0.224, 0.225])
            ])

        def __len__(self):
            return len(self.images)

        def __getitem__(self, idx):
            img = self.images[idx]
            if self.transform:
                img = self.transform(img)
            tab_data = torch.tensor(self.tab_features[idx], dtype=torch.float32)
            price = torch.tensor([self.prices[idx]], dtype=torch.float32)
            return (img, tab_data), price

    class MultiInputPricePredictor(nn.Module):
        def __init__(self, tab_dim):
            super().__init__()
            self.image_branch = nn.Sequential(
                nn.Conv2d(3, 16, kernel_size=3, padding=1),
                nn.ReLU(),
                nn.MaxPool2d(2),
                nn.Conv2d(16, 32, kernel_size=3, padding=1),
                nn.ReLU(),
                nn.MaxPool2d(2),
                nn.Conv2d(32, 64, kernel_size=3, padding=1),
                nn.ReLU(),
                nn.MaxPool2d(2),
                nn.Flatten()
            )
            self.tab_branch = nn.Sequential(
                nn.Linear(tab_dim, 32),
                nn.ReLU()
            )
            self.regressor = nn.Sequential(
                nn.Linear(64 * 4 * 4 + 32, 128),
                nn.ReLU(),
                nn.Dropout(0.3),
                nn.Linear(128, 1)
            )

        def forward(self, x):
            img, tab = x
            f_img = self.image_branch(img)
            f_tab = self.tab_branch(tab)
            combined = torch.cat((f_img, f_tab), dim=1)
            return self.regressor(combined)

    def train_model(self, epochs=30):
        tab_dim = self.train_dataset.tab_features.shape[1]
        model = self.MultiInputPricePredictor(tab_dim).to(self.device)
        criterion = nn.MSELoss()  # Log-Skala
        optimizer = optim.Adam(model.parameters(), lr=self.lr)

        best_val_loss = float('inf')
        patience_counter = 0
        train_losses, val_losses = [], []

        for epoch in range(epochs):
            model.train()
            running = 0.0
            for (imgs, tabs), yb in self.train_loader:
                imgs, tabs, yb = imgs.to(self.device), tabs.to(self.device), yb.to(self.device)
                optimizer.zero_grad()
                out = model((imgs, tabs))
                loss = criterion(out, yb)
                loss.backward()
                optimizer.step()
                running += loss.item() * imgs.size(0)
            tr_loss = running / len(self.train_loader.dataset)
            train_losses.append(tr_loss)

            model.eval()
            running = 0.0
            with torch.no_grad():
                for (imgs, tabs), yb in self.test_loader:
                    imgs, tabs, yb = imgs.to(self.device), tabs.to(self.device), yb.to(self.device)
                    out = model((imgs, tabs))
                    loss = criterion(out, yb)
                    running += loss.item() * imgs.size(0)
            va_loss = running / len(self.test_loader.dataset)
            val_losses.append(va_loss)

            print(f"Epoch {epoch+1}/{epochs} - Train Loss (log): {tr_loss:.4f} | Val Loss (log): {va_loss:.4f}")

            if va_loss < best_val_loss:
                best_val_loss = va_loss
                patience_counter = 0
                torch.save(model.state_dict(), "best_model.pth")
            else:
                patience_counter += 1
                if patience_counter >= self.patience:
                    print("Early stopping")
                    break

        self.model = model
        return train_losses, val_losses

    def evaluate_loader(self, loader):
        preds, targets = [], []
        self.model.eval()
        with torch.no_grad():
            for (imgs, tabs), yb in loader:
                imgs, tabs = imgs.to(self.device), tabs.to(self.device)
                out = self.model((imgs, tabs))
                preds.extend(out.cpu().numpy().flatten())
                targets.extend(yb.cpu().numpy().flatten())
        return np.array(preds), np.array(targets)


In [20]:
csv_path = "data/listings.csv.gz"

trainer = AirbnbPreprocessorAndTrainer(csv_path, nrows=10000, image_size=32, batch_size=64, lr=0.001, patience=5, seed=SEED)
trainer.preprocess()
trainer.process_images()
trainer.prepare_tensors()

train_losses, val_losses = trainer.train_model(epochs=30)
trainer.model.load_state_dict(torch.load("best_model.pth", map_location=trainer.device))

preds_train, y_train_log = trainer.evaluate_loader(trainer.train_loader)
preds_test,  y_test_log  = trainer.evaluate_loader(trainer.test_loader)

rmse_tr, mae_tr, medae_tr, r2_tr = sklearn_metrics_log(preds_train, y_train_log)
rmse_te, mae_te, medae_te, r2_te = sklearn_metrics_log(preds_test,  y_test_log)

results = pd.DataFrame([
    {"Split": "Train", "RMSE (log)": rmse_tr, "MAE (log)": mae_tr, "MedAE (log)": medae_tr, "R² (log)": r2_tr},
    {"Split": "Test",  "RMSE (log)": rmse_te, "MAE (log)": mae_te, "MedAE (log)": medae_te, "R² (log)": r2_te}
])
results


Verwende Gerät: cpu


AttributeError: 'AirbnbPreprocessorAndTrainer' object has no attribute 'num_cols'

In [21]:
plt.figure(figsize=(6,4))
plt.plot(train_losses, label="Train Loss (log)")
plt.plot(val_losses, label="Val Loss (log)")
plt.xlabel("Epoch")
plt.ylabel("Loss")
plt.title("Training/Validation Loss (log)")
plt.legend()
plt.grid(True)
plt.tight_layout()
plt.show()


NameError: name 'train_losses' is not defined

<Figure size 600x400 with 0 Axes>

In [22]:
# Residuen (log)
res_train = preds_train - y_train_log
res_test  = preds_test  - y_test_log

fig, ax = plt.subplots(1, 2, figsize=(10,4))
ax.hist(res_train, bins=30, alpha=0.7)
ax.set_title("Residuen (log) – Train")
ax.set_xlabel("Pred - True (log)")
ax.set_ylabel("Häufigkeit")
ax.grid(True)

ax[1].hist(res_test, bins=30, alpha=0.7, color="orange")
ax[1].set_title("Residuen (log) – Test")
ax[1].set_xlabel("Pred - True (log)")
ax[1].set_ylabel("Häufigkeit")
ax[1].grid(True)
plt.tight_layout()
plt.show()

# Histogramme der Vorhersagen und Wahrheiten (log)
fig, ax = plt.subplots(1, 2, figsize=(10,4))
ax.hist(y_train_log, bins=30, alpha=0.6, label="True", density=True)
ax.hist(preds_train, bins=30, alpha=0.6, label="Pred", density=True)
ax.set_title("Train: True vs Pred (log)")
ax.legend()
ax.grid(True)

ax[1].hist(y_test_log, bins=30, alpha=0.6, label="True", density=True)
ax[1].hist(preds_test, bins=30, alpha=0.6, label="Pred", density=True)
ax[1].set_title("Test: True vs Pred (log)")
ax[1].legend()
ax[1].grid(True)
plt.tight_layout()
plt.show()

# True vs Predicted Scatter (log)
lims = [min(y_test_log.min(), preds_test.min()), max(y_test_log.max(), preds_test.max())]
plt.figure(figsize=(5,5))
plt.scatter(y_test_log, preds_test, alpha=0.4)
plt.plot(lims, lims, 'r--', label='Ideal')
plt.xlim(lims); plt.ylim(lims)
plt.xlabel("True (log)"); plt.ylabel("Pred (log)")
plt.title("True vs Predicted (log) – Test")
plt.legend(); plt.grid(True); plt.tight_layout(); plt.show()


NameError: name 'preds_train' is not defined

In [ ]:
def permutation_feature_importance_log(trainer, n_repeats=5):
    base_rmse = mean_squared_error(*trainer.evaluate_loader(trainer.test_loader)[::-1], squared=False)
    imgs_base = trainer.test_dataset.images
    tabs_base = trainer.test_dataset.tab_features
    y_base = trainer.test_dataset.prices
    n_features = tabs_base.shape[1]
    importances = []

    for j in range(n_features):
        deltas = []
        for _ in range(n_repeats):
            perm_tabs = tabs_base.copy()
            perm_tabs[:, j] = np.random.permutation(perm_tabs[:, j])
            perm_ds = trainer.AirbnbDataset(imgs_base, perm_tabs, y_base)
            perm_loader = DataLoader(perm_ds, batch_size=trainer.batch_size, shuffle=False)
            preds_p, y_p = trainer.evaluate_loader(perm_loader)
            rmse_p = mean_squared_error(y_p, preds_p, squared=False)
            deltas.append(rmse_p - base_rmse)
        importances.append(np.mean(deltas))

    fi = pd.DataFrame({"Feature": trainer.feature_names, "ΔRMSE (log)": importances})
    fi = fi.sort_values("ΔRMSE (log)", ascending=False).reset_index(drop=True)
    return fi

fi_df = permutation_feature_importance_log(trainer, n_repeats=5)
fi_df.head(10)
